# Pré-processamento de dados

**Objetivo:** partir de dados sujos (escalas diferentes, faltantes, categorias em texto), montar um `ColumnTransformer` completo e medir o impacto da padronização num modelo baseado em distância.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta do curso (idêntica ao site)
INK, PAPER = "#1a1a1a", "#fffdf8"
BLUE, RED, GREEN, MUTED = "#3266ad", "#c0392b", "#1a7a4a", "#6b6457"

plt.rcParams.update({
    "font.family": "serif", "font.size": 12,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER,
    "axes.edgecolor": "#b9ad95", "axes.grid": True,
    "grid.color": "#e2d9c4", "grid.linewidth": 0.7,
    "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
})
rng = np.random.default_rng(42)

## Um conjunto de dados sujo

In [ ]:
df = pd.DataFrame({
    'idade':     [34, 51, np.nan, 62, 45, 29],
    'colesterol':[190, 240, 210, np.nan, 260, 175],
    'sexo':      ['F', 'M', 'M', 'F', np.nan, 'F'],
    'risco':     [0, 1, 0, 1, 1, 0],
})
df

## Pipeline: imputação + escala (num) e imputação + one-hot (cat)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num = ['idade', 'colesterol']
cat = ['sexo']
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('oh', OneHotEncoder())])
pre = ColumnTransformer([('num', num_pipe, num), ('cat', cat_pipe, cat)])
Xt = pre.fit_transform(df[num + cat])
print('matriz processada:\n', np.round(Xt, 2))

## Padronização importa para k-NN?

Comparamos a acurácia (validação cruzada) com e sem escala.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer

Xbc, ybc = load_breast_cancer(return_X_y=True)
sem = KNeighborsClassifier()
com = Pipeline([('sc', StandardScaler()), ('knn', KNeighborsClassifier())])
print('sem escala:', round(cross_val_score(sem, Xbc, ybc, cv=5).mean(), 3))
print('com escala:', round(cross_val_score(com, Xbc, ybc, cv=5).mean(), 3))

## Exercícios

**1.** De quanto foi o ganho da padronização no k-NN? Por que ele aparece justamente num modelo de distância?

**2.** Troque `StandardScaler` por `MinMaxScaler`. O resultado muda muito?

In [ ]:
# @title Solução (clique para revelar)
from sklearn.preprocessing import MinMaxScaler
mm = Pipeline([('sc', MinMaxScaler()), ('knn', KNeighborsClassifier())])
print('min-max:', round(cross_val_score(mm, Xbc, ybc, cv=5).mean(), 3))
# O ganho aparece porque o k-NN soma distâncias entre características;
# sem escala, as de maior amplitude dominam. Standard e MinMax costumam
# dar resultados parecidos aqui.